## Setup
Set `REPO_URL` to your GitHub repo, **or** leave it empty and attach the repo (uploaded as a Kaggle dataset) as an input.
Accelerator: **GPU T4 x2**. Internet: **On**.

In [ ]:
import os, glob, shutil, subprocess, json, time
REPO_URL = "https://github.com/areejfatima2998-creator/small-model-alignment.git"
WORK = "/kaggle/working"
REPO = f"{WORK}/small-model-alignment"
if not os.path.exists(REPO):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
    else:
        src = [p for p in glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True) if "/outputs/" not in p]
        assert src, "Attach the repo as a Kaggle dataset or set REPO_URL"
        shutil.copytree(os.path.dirname(src[0]), REPO, ignore=shutil.ignore_patterns("outputs"))
os.chdir(REPO)
os.environ["PYTHONPATH"] = REPO
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(os.getcwd())

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
!python -m pytest -q tests/

## Task 2 baseline + Task 3 SFT
Baseline generation runs on **cuda:1 in the background** while SFT trains on **cuda:0**.
Expected: ~2 h. When done: **Save Version -> Save & Run All (Commit)** so outputs persist.

In [ ]:
os.makedirs("outputs/generations", exist_ok=True)
# The fixed AlpacaEval subset (300 prompts, seed 42) is committed in eval_sets/; this only re-verifies it.
!python scripts/make_alpaca_subset.py --output /tmp/alpaca_check.json && cmp /tmp/alpaca_check.json eval_sets/alpaca_eval_300_seed42.json && echo "subset identical"

In [ ]:
base_gen = subprocess.Popen(
    "python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --eval_set eval_sets/alpaca_eval_300_seed42.json "
    "--generator qwen2.5-0.5b-base --output outputs/generations/base_alpaca.json --device cuda:1 --max_new_tokens 384 --batch_size 16 --repetition_penalty 1.1 && "
    "python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --eval_set eval_sets/hh_rlhf_eval_300_seed42.json "
    "--generator qwen2.5-0.5b-base --output outputs/generations/base_hh.json --device cuda:1 --max_new_tokens 384 --batch_size 16 --repetition_penalty 1.1",
    shell=True, stdout=open("outputs/generations/base_gen.log", "w"), stderr=subprocess.STDOUT)
print("baseline generation started, pid", base_gen.pid)

In [ ]:
!python scripts/train_sft.py --output_dir outputs/sft --device cuda:0

In [ ]:
base_gen.wait(); print("baseline exit code", base_gen.returncode)
!tail -5 outputs/generations/base_gen.log

### Sanity check: read these 3 SFT outputs (prompt format, EOS, length look sane?)

In [ ]:
!python scripts/generate.py --model_path outputs/sft/best --eval_set eval_sets/alpaca_eval_300_seed42.json --generator qwen2.5-0.5b-sft --output /tmp/sanity.json --limit 3 --max_new_tokens 384 --batch_size 16 --repetition_penalty 1.1

In [ ]:
job = subprocess.Popen("python scripts/generate.py --model_path outputs/sft/best --eval_set eval_sets/alpaca_eval_300_seed42.json --generator qwen2.5-0.5b-sft --output outputs/generations/sft_alpaca.json --device cuda:0 --max_new_tokens 384 --batch_size 16 --repetition_penalty 1.1", shell=True, stdout=open("outputs/generations/sft_alpaca.log", "w"), stderr=subprocess.STDOUT)
!python scripts/generate.py --model_path outputs/sft/best --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator qwen2.5-0.5b-sft --output outputs/generations/sft_hh.json --device cuda:1 --max_new_tokens 384 --batch_size 16 --repetition_penalty 1.1
job.wait(); assert job.returncode == 0, f"background job failed with exit code {job.returncode}"
!tail -3 outputs/generations/sft_alpaca.log

In [ ]:
!python scripts/plot_metrics.py --sft outputs/sft/metrics.jsonl --out_dir outputs/plots
!cd outputs && zip -qr ../sft_run_outputs.zip . -x "*.safetensors" && cd .. && ls -la outputs/* sft_run_outputs.zip